Feature Engineering

This notebook prepares the cleaned NY HMDA 2015 data for modeling. It drops leakage and unhelpful columns, adds a few row-level features, and splits the data into train and test. It then imputes, one-hot encodes and scales the features, fitting every step on the training split only.

**Run the two earlier notebooks first.** This one continues where they leave off:
- `Duplicates and Outliers.ipynb` saves `../data/ny_hmda_2015_dedup_outliers.csv`, with duplicates and `*_name` columns removed and income / loan amount capped at the 1st / 99th percentiles. That file is this notebook's input.
- `Data Understanding.ipynb` defines the target `is_approved`. It's rebuilt below with one change: purchased loans (`action_taken` 6) are removed instead of counted as approvals.

**Outputs left in memory for model setup:**
- `X_train_prep`, `X_test_prep`: imputed, encoded and scaled features (fully numeric, no NaNs)
- `y_train`, `y_test`: the target
- `protected_train`, `protected_test`: protected attributes for fairness evaluation, aligned with `X_train_prep` / `X_test_prep` by index
- `cat_imputer`, `num_imputer`, `encoder`, `scaler`: the fitted objects, for transforming new data the same way (after `add_row_features()`)
- `X_train`, `X_test`: the split data *before* imputing, encoding and scaling

In [8]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

### Settings

- `USE_PROTECTED_AS_FEATURES`: when `True`, sex, ethnicity and race (applicant and co-applicant) are one-hot encoded as model features. The default is `False`, which keeps them in the data for fairness evaluation only.

In [9]:
USE_PROTECTED_AS_FEATURES = True
RANDOM_STATE = 42
TARGET = 'is_approved'

## Load the output of Duplicates and Outliers

In [10]:
input_filename = os.path.join(os.getcwd(), '..', 'data', 'ny_hmda_2015_dedup_outliers.csv')
if not os.path.exists(input_filename):
    raise FileNotFoundError(f'{input_filename} not found. Run "Duplicates and Outliers.ipynb" first.')

df_cleaned = pd.read_csv(input_filename, low_memory=False)
df_cleaned.shape
df_cleaned

,action_taken,agency_code,agency_abbr,applicant_ethnicity,applicant_income_000s,applicant_race_1,applicant_race_2,applicant_race_3,applicant_race_4,applicant_race_5,...,state_code,state_abbr,hud_median_family_income,loan_amount_000s,number_of_1_to_4_family_units,number_of_owner_occupied_units,minority_population,population,rate_spread,tract_to_msamd_income
0,1,9,CFPB,2,97.0,5,NaN,NaN,NaN,NaN,...,36,NY,109000.0,187,363.0,1817.0,21.139999,5870.0,NaN,109.459999
1,1,9,CFPB,2,200.0,5,NaN,NaN,NaN,NaN,...,36,NY,71300.0,460,53.0,256.0,45.959999,3512.0,NaN,160.600006
2,1,7,HUD,2,NaN,3,NaN,NaN,NaN,NaN,...,36,NY,71300.0,296,2745.0,2586.0,38.990002,8357.0,NaN,134.820007
3,1,9,CFPB,2,202.0,5,NaN,NaN,NaN,NaN,...,36,NY,109000.0,770,1879.0,2147.0,7.350000,6642.0,NaN,165.830002
4,1,9,CFPB,2,255.0,5,NaN,NaN,NaN,NaN,...,36,NY,109000.0,648,835.0,676.0,30.059999,2339.0,NaN,133.300003
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
439649,6,9,CFPB,4,82.0,7,NaN,NaN,NaN,NaN,...,36,NY,71300.0,510,948.0,391.0,43.790001,2811.0,NaN,85.279999
439650,6,7,HUD,4,NaN,7,NaN,NaN,NaN,NaN,...,36,NY,69000.0,98,2287.0,1783.0,6.130000,5984.0,NaN,111.459999
439651,6,7,HUD,4,NaN,7,NaN,NaN,NaN,NaN,...,36,NY,69700.0,128,1814.0,1655.0,4.140000,5093.0,NaN,129.479996
439652,6,9,CFPB,4,95.0,7,NaN,NaN,NaN,NaN,...,36,NY,109000.0,168,1423.0,1480.0,19.969999,4997.0,NaN,171.649994


## Build the target `is_approved`

This follows `Data Understanding.ipynb`, except that purchased loans are removed rather than counted as approvals:
- Remove applications that were withdrawn (`action_taken` 4) or closed for incompleteness (5), since neither is a lending decision.
- Remove loans the institution purchased from another lender (6). The reporting institution didn't decide whether to approve them, so they don't belong in an approval model.
- Approved (1): loan originated (1), approved but not accepted (2).
- Denied (0): application denied (3), preapproval request denied (7).
- Then drop `action_taken`, since it *is* the label.

In [11]:
df_cleaned = df_cleaned[~df_cleaned['action_taken'].isin([4, 5, 6])]
df_cleaned['is_approved'] = df_cleaned['action_taken'].isin([1, 2]).astype(int)
df_cleaned = df_cleaned.drop(columns=['action_taken'])

print('Shape:', df_cleaned.shape)
print(f'Approval rate: {df_cleaned["is_approved"].mean():.3f}')

Shape: (321935, 47)
Approval rate: 0.752


## Step 1: Work on a copy

Every change below is made on `df`, so `df_cleaned` (the cleaned data plus the target) stays untouched.

In [12]:
df = df_cleaned.copy()
print('Starting shape:', df.shape)

Starting shape: (321935, 47)


## Steps 2–3: Drop leakage and unhelpful columns

**Leakage.** These values are only known *after* the lending decision, so a model that used them would effectively be seeing the answer:
- `denial_reason_1`–`3`: filled in only for denied applications
- `rate_spread`: reported only for originated loans
- `purchaser_type`: non-zero only for loans that were made and then sold, so it reveals approval
- `action_taken`: the raw label (already removed when building the target; listed here just in case)

**Not useful for modeling:**
- Constant: `as_of_year` (always 2015), `state_code` (always NY) and `application_date_indicator` (`0` on every row but one, since its other common value, `2`, marked purchased loans)
- Mostly empty: `edit_status` (81% missing), `applicant_race_2`–`5` and `co_applicant_race_2`–`5` (over 99% missing)
- ID-like: `sequence_number`, `respondent_id` and `census_tract_number` have too many unique values to generalize
- Redundant: `county_code`, since counties roll up into the metro areas already captured by `msamd`

**Leftover text columns.** `agency_abbr` or `state_abbr`. Those only repeat `agency_code` and `state_code`, so they're dropped here too, along with any `*_name` column that's still around.

Each list is filtered to the columns that actually exist, so this cell still works once the upstream scripts are fixed.

In [13]:
LEAKAGE_COLS = ['denial_reason_1', 'denial_reason_2', 'denial_reason_3',
                'rate_spread', 'purchaser_type', 'action_taken']
CONSTANT_COLS = ['as_of_year', 'state_code', 'application_date_indicator']
MOSTLY_EMPTY_COLS = (['edit_status']
                     + [f'applicant_race_{i}' for i in range(2, 6)]
                     + [f'co_applicant_race_{i}' for i in range(2, 6)])
ID_LIKE_COLS = ['sequence_number', 'respondent_id', 'census_tract_number']
REDUNDANT_COLS = ['county_code']
TEXT_COLS = [c for c in df.columns if '_name' in c or c.endswith('_abbr')]

drop_groups = {
    'leakage': LEAKAGE_COLS,
    'constant': CONSTANT_COLS,
    'mostly empty': MOSTLY_EMPTY_COLS,
    'ID-like': ID_LIKE_COLS,
    'redundant': REDUNDANT_COLS,
    'text labels': TEXT_COLS,
}
for group, cols in drop_groups.items():
    present = [c for c in cols if c in df.columns]
    print(f'Dropping {group} ({len(present)}): {present}')
    df = df.drop(columns=present)

print('Shape after drops:', df.shape)

Dropping leakage (5): ['denial_reason_1', 'denial_reason_2', 'denial_reason_3', 'rate_spread', 'purchaser_type']
Dropping constant (3): ['as_of_year', 'state_code', 'application_date_indicator']
Dropping mostly empty (9): ['edit_status', 'applicant_race_2', 'applicant_race_3', 'applicant_race_4', 'applicant_race_5', 'co_applicant_race_2', 'co_applicant_race_3', 'co_applicant_race_4', 'co_applicant_race_5']
Dropping ID-like (3): ['sequence_number', 'respondent_id', 'census_tract_number']
Dropping redundant (1): ['county_code']
Dropping text labels (2): ['agency_abbr', 'state_abbr']
Shape after drops: (321935, 24)


## Step 4: Row-level features

Each of these uses only values from its own row, so creating them before the train/test split can't leak information between the splits.

- `income_missing`: 1 if `applicant_income_000s` is missing. It's created *before* imputation so the model can tell a filled-in median income apart from a real one.
- `has_co_applicant`: 1 unless `co_applicant_sex == 5` ("No co-applicant").
- `loan_to_income`: `loan_amount_000s / applicant_income_000s`. It's left as NaN when income is missing or 0 and imputed later, from the training split only.
- `log_*`: `np.log1p` of income, loan amount and loan-to-income. Income and loan amount were already capped at the 1st / 99th percentiles in Duplicates and Outliers, but all three are still right-skewed. The log reduces that skew so a few large values don't dominate the logistic regression. The raw columns stay in the DataFrame for reference and for fairness slicing, but they aren't model inputs.

The logic lives in `add_row_features()` so the same steps can be applied to new data.

In [14]:
LOG_SOURCE_COLS = ['applicant_income_000s', 'loan_amount_000s', 'loan_to_income']
LOG_COLS = [f'log_{c}' for c in LOG_SOURCE_COLS]

def add_row_features(frame):
    out = frame.copy()
    income = out['applicant_income_000s']
    out['income_missing'] = income.isna().astype(int)
    out['has_co_applicant'] = (out['co_applicant_sex'] != 5).astype(int)
    out['loan_to_income'] = out['loan_amount_000s'] / income.replace(0, np.nan)
    for src, dst in zip(LOG_SOURCE_COLS, LOG_COLS):
        out[dst] = np.log1p(out[src])
    return out

df = add_row_features(df)
df[['income_missing', 'has_co_applicant'] + LOG_SOURCE_COLS + LOG_COLS].describe().T

,count,mean,std,min,25%,50%,75%,max
income_missing,321935.0,0.076158,0.265252,0.000000,0.000000,0.000000,0.000000,1.000000
has_co_applicant,321935.0,0.431541,0.495292,0.000000,0.000000,0.000000,1.000000,1.000000
applicant_income_000s,297417.0,127.959555,141.280003,17.000000,57.000000,89.000000,140.000000,1000.000000
loan_amount_000s,321935.0,288.914532,344.477776,4.000000,95.000000,199.000000,360.000000,2280.000000
loan_to_income,297417.0,2.417035,2.247909,0.004000,1.220779,2.111111,3.142857,134.117647
log_applicant_income_000s,297417.0,4.541800,0.740536,2.890372,4.060443,4.499810,4.948760,6.908755
log_loan_amount_000s,321935.0,5.109265,1.195536,1.609438,4.564348,5.298317,5.888878,7.732369
log_loan_to_income,297417.0,1.097043,0.501991,0.003992,0.797858,1.134980,1.421386,4.906146


## Step 5: Train/test split

The data is split 80/20, stratified on `is_approved` so both splits keep the same approval rate.

**From here on, every imputer, encoder and scaler is fit on `X_train` only.** `X_test` is only transformed with the values learned from training, so no test-set information leaks into the preprocessing.

In [15]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

print('Train:', X_train.shape, f'approval rate {y_train.mean():.3f}')
print('Test: ', X_test.shape, f'approval rate {y_test.mean():.3f}')

Train: (257548, 29) approval rate 0.752
Test:  (64387, 29) approval rate 0.752


## Step 6: Column groups

Each model input belongs to one group, and each group is prepared differently in the next steps:

| Group | Columns | Impute (Step 7) | Transform |
| - | - | - | - |
| Categorical | `agency_code`, `loan_type`, `property_type`, `loan_purpose`, `owner_occupancy`, `preapproval`, `lien_status`, `hoepa_status`, `msamd` | `-1` (a category of its own) | One-hot encode (Step 8) |
| Numeric | the 3 `log_*` columns, `hud_median_family_income`, `tract_to_msamd_income`, `minority_population`, `population`, `number_of_owner_occupied_units`, `number_of_1_to_4_family_units` | Training median | Standardize (Step 9) |
| Binary | `income_missing`, `has_co_applicant` | Not needed | None (already 0/1) |

Columns that aren't in any group, such as the raw income and loan columns, aren't model inputs.

**Protected attributes** (`applicant_sex`, `applicant_ethnicity`, `applicant_race_1` and their co-applicant versions) are added to the categorical group only when `USE_PROTECTED_AS_FEATURES = True`. Either way, they're kept in `protected_train` / `protected_test` for the fairness evaluation.

In [16]:
CATEGORICAL_COLS = ['agency_code', 'loan_type', 'property_type', 'loan_purpose',
                    'owner_occupancy', 'preapproval', 'lien_status', 'hoepa_status', 'msamd']
PROTECTED_COLS = ['applicant_sex', 'applicant_ethnicity', 'applicant_race_1',
                  'co_applicant_sex', 'co_applicant_ethnicity', 'co_applicant_race_1']
NUMERIC_COLS = LOG_COLS + ['hud_median_family_income', 'tract_to_msamd_income',
                           'minority_population', 'population',
                           'number_of_owner_occupied_units', 'number_of_1_to_4_family_units']
BINARY_COLS = ['income_missing', 'has_co_applicant']

cat_cols = CATEGORICAL_COLS + (PROTECTED_COLS if USE_PROTECTED_AS_FEATURES else [])

protected_train = X_train[PROTECTED_COLS]
protected_test = X_test[PROTECTED_COLS]

print('Protected attributes used as features:', USE_PROTECTED_AS_FEATURES)
print(f'Model inputs: {len(cat_cols)} categorical, {len(NUMERIC_COLS)} numeric, {len(BINARY_COLS)} binary columns')

Protected attributes used as features: True
Model inputs: 15 categorical, 9 numeric, 2 binary columns


## Step 7: Impute missing values

- Categorical: a missing code becomes `-1`, a category of its own. `msamd` is missing for properties outside a metro area, so `-1` effectively means "non-metro".
- Numeric: a missing value is filled with the **training** median. `loan_to_income` is missing whenever income is, and the census columns are missing for under 1% of rows.

Each imputer learns from `X_train` (`fit_transform`) and is then applied to `X_test` (`transform`).

In [17]:
missing = X_train[cat_cols + NUMERIC_COLS].isna().sum()
print('Missing values in the training split:')
print(missing[missing > 0].to_string())

cat_imputer = SimpleImputer(strategy='constant', fill_value=-1)
num_imputer = SimpleImputer(strategy='median')

train_cat = pd.DataFrame(cat_imputer.fit_transform(X_train[cat_cols]),
                         columns=cat_cols, index=X_train.index).astype(int)
test_cat = pd.DataFrame(cat_imputer.transform(X_test[cat_cols]),
                        columns=cat_cols, index=X_test.index).astype(int)

train_num = pd.DataFrame(num_imputer.fit_transform(X_train[NUMERIC_COLS]),
                         columns=NUMERIC_COLS, index=X_train.index)
test_num = pd.DataFrame(num_imputer.transform(X_test[NUMERIC_COLS]),
                        columns=NUMERIC_COLS, index=X_test.index)

Missing values in the training split:
msamd                             23261
log_applicant_income_000s         19638
log_loan_to_income                19638
hud_median_family_income            720
tract_to_msamd_income               781
minority_population                 735
population                          735
number_of_owner_occupied_units      961
number_of_1_to_4_family_units      1046


## Step 8: One-hot encode the categorical columns

HMDA codes are categories, not quantities (for example, `loan_type` 3 isn't "more" than 1). So each code becomes its own 0/1 column, named `<column>_<code>`, e.g. `loan_purpose_1`.

The encoder learns the list of codes from the training split. `handle_unknown='ignore'` means a code that shows up only in the test set becomes all zeros instead of raising an error.

In [18]:
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False, dtype=int)

train_cat_enc = pd.DataFrame(encoder.fit_transform(train_cat),
                             columns=encoder.get_feature_names_out(), index=X_train.index)
test_cat_enc = pd.DataFrame(encoder.transform(test_cat),
                            columns=encoder.get_feature_names_out(), index=X_test.index)

print(f'{len(cat_cols)} categorical columns -> {train_cat_enc.shape[1]} one-hot columns')

15 categorical columns -> 75 one-hot columns


## Step 9: Scale the numeric columns

`StandardScaler` rescales each numeric column to mean 0 and standard deviation 1, using the **training** mean and standard deviation. Without it, columns with large values (like `population`) would outweigh columns with small ones (like the logged ratios), which matters for a regularized logistic regression.

In [19]:
scaler = StandardScaler()

train_num_scaled = pd.DataFrame(scaler.fit_transform(train_num),
                                columns=NUMERIC_COLS, index=X_train.index)
test_num_scaled = pd.DataFrame(scaler.transform(test_num),
                               columns=NUMERIC_COLS, index=X_test.index)

train_num_scaled.describe().T[['mean', 'std']].round(2)

,mean,std
log_applicant_income_000s,-0.0,1.0
log_loan_amount_000s,-0.0,1.0
log_loan_to_income,-0.0,1.0
hud_median_family_income,0.0,1.0
tract_to_msamd_income,0.0,1.0
minority_population,0.0,1.0
population,-0.0,1.0
number_of_owner_occupied_units,-0.0,1.0
number_of_1_to_4_family_units,0.0,1.0


## Step 10: Combine into the final feature matrices

The encoded, scaled and binary columns are joined side by side (they all share the same row index). The checks confirm that no NaNs are left and that train and test have the same columns.

In [20]:
X_train_prep = pd.concat([train_cat_enc, train_num_scaled, X_train[BINARY_COLS]], axis=1)
X_test_prep = pd.concat([test_cat_enc, test_num_scaled, X_test[BINARY_COLS]], axis=1)

assert not X_train_prep.isna().any().any(), 'NaNs left in X_train_prep'
assert not X_test_prep.isna().any().any(), 'NaNs left in X_test_prep'
assert list(X_train_prep.columns) == list(X_test_prep.columns)

print('X_train_prep:', X_train_prep.shape)
print('X_test_prep: ', X_test_prep.shape)
X_train_prep.columns

X_train_prep: (257548, 86)
X_test_prep:  (64387, 86)


Index(['agency_code_1', 'agency_code_2', 'agency_code_3', 'agency_code_5',
       'agency_code_7', 'agency_code_9', 'loan_type_1', 'loan_type_2',
       'loan_type_3', 'loan_type_4', 'property_type_1', 'property_type_2',
       'property_type_3', 'loan_purpose_1', 'loan_purpose_2', 'loan_purpose_3',
       'owner_occupancy_1', 'owner_occupancy_2', 'owner_occupancy_3',
       'preapproval_1', 'preapproval_2', 'preapproval_3', 'lien_status_1',
       'lien_status_2', 'lien_status_3', 'hoepa_status_1', 'hoepa_status_2',
       'msamd_-1', 'msamd_10580', 'msamd_13780', 'msamd_15380', 'msamd_20524',
       'msamd_21300', 'msamd_24020', 'msamd_27060', 'msamd_28740',
       'msamd_35004', 'msamd_35614', 'msamd_40380', 'msamd_45060',
       'msamd_46540', 'msamd_48060', 'applicant_sex_1', 'applicant_sex_2',
       'applicant_sex_3', 'applicant_sex_4', 'applicant_ethnicity_1',
       'applicant_ethnicity_2', 'applicant_ethnicity_3',
       'applicant_ethnicity_4', 'applicant_race_1_1', 'applica